# AGILAB data and pipeline-graph launch in Colab

This notebook shows two built-in AGILAB paths from Google Colab:

- `execution_pandas_project` for a data-worker-style workload.
- `uav_relay_queue_project` for a DAG-shaped pipeline with topology and queue artifacts.

It clones the AGILAB repository, installs it in editable mode, installs
the worker the first time, and runs both paths locally through `AgiEnv`
and `AGI.run(...)`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThalesGroup/agilab/blob/main/examples/notebook_quickstart/agi_core_colab_data_dag.ipynb)


In [ ]:
!if [ -d /content/agilab/.git ]; then cd /content/agilab && git pull --ff-only; else rm -rf /content/agilab && git clone --depth 1 https://github.com/ThalesGroup/agilab.git /content/agilab; fi
!python -m pip uninstall -y agilab agi-cluster agi-node agi-env >/dev/null 2>&1 || true
!pip install -q -e /content/agilab/src/agilab/core/agi-env -e /content/agilab/src/agilab/core/agi-node -e /content/agilab/src/agilab/core/agi-cluster -e /content/agilab

import os
import pathlib
from io import UnsupportedOperation as IOUnsupportedOperation
from pathlib import Path
import sys

if not hasattr(pathlib, "UnsupportedOperation"):
    pathlib.UnsupportedOperation = IOUnsupportedOperation

for name in [module for module in list(sys.modules) if module == "agi_cluster" or module.startswith("agi_cluster.") or module == "agi_env" or module.startswith("agi_env.")]:
    del sys.modules[name]
import importlib
importlib.invalidate_caches()

REPO_ROOT = Path("/content/agilab")
CORE_NODE_SRC = REPO_ROOT / "src" / "agilab" / "core" / "agi-node" / "src"
CORE_CLUSTER_SRC = REPO_ROOT / "src" / "agilab" / "core" / "agi-cluster" / "src"
CORE_ENV_SRC = REPO_ROOT / "src" / "agilab" / "core" / "agi-env" / "src"
LOCAL_CORE_PATHS = [REPO_ROOT / "src", CORE_NODE_SRC, CORE_ENV_SRC, CORE_CLUSTER_SRC]
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(path) for path in LOCAL_CORE_PATHS]
    + ([os.environ["PYTHONPATH"]] if os.environ.get("PYTHONPATH") else [])
)
for path in LOCAL_CORE_PATHS:
    sys.path.insert(0, str(path))
from agi_cluster.agi_distributor import AGI
from agi_env import AgiEnv

APPS_PATH = REPO_ROOT / "src" / "agilab" / "apps" / "builtin"

print("Repo root:", REPO_ROOT)
print("Apps path:", APPS_PATH)
print("execution_pandas_project exists:", (APPS_PATH / "execution_pandas_project").is_dir())
print("uav_relay_queue_project exists:", (APPS_PATH / "uav_relay_queue_project").is_dir())


In [ ]:
async def run_app(app_name: str):
    app_env = AgiEnv(apps_path=APPS_PATH, app=app_name, verbose=0)
    worker_venv = Path.home() / "wenv" / f"{app_env.target}_worker" / ".venv"
    if not worker_venv.exists():
        print(f"Installing worker for {app_name}...")
        await AGI.install(
            app_env,
            scheduler="127.0.0.1",
            workers={"127.0.0.1": 1},
            modes_enabled=0,
        )
    result = await AGI.run(
        app_env,
        scheduler="127.0.0.1",
        workers={"127.0.0.1": 1},
        mode=0,
    )
    log_root = Path.home() / "log" / "execute" / app_env.target
    print(f"\n=== {app_name} ===")
    print("Target:", app_env.target)
    print("Log root:", log_root)
    if log_root.exists():
        print("Files:", sorted(path.name for path in log_root.iterdir())[:20])
    return result


In [ ]:
data_result = await run_app("execution_pandas_project")
data_result


In [ ]:
dag_result = await run_app("uav_relay_queue_project")
dag_result
